In [ ]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances


# ============================================================
# Configuration
# ============================================================

N_BOOTSTRAP = 10
CONFIDENCE = 0.95
RANDOM_SEED = 11

TOP_NS = [1, 5, 10, 30]


# ============================================================
# Evaluation
# ============================================================

def eval(
    gallery_df,
    gallery_set_representations,
    test_set_representations,
    test_synd_ids,
    calculate_bootstrap=False,
    n_bootstrap=N_BOOTSTRAP,
    confidence=CONFIDENCE,
    seed=RANDOM_SEED
):
    """
    Evaluate the ensemble for Top-1, Top-5, Top-10 and Top-30.

    The ensemble itself is unchanged from the original implementation:
        1. Calculate cosine distance for each model.
        2. Average the distances over the models.
        3. Rank gallery images.
        4. Remove duplicate syndrome IDs while preserving rank.
        5. Calculate Top-N performance.

    The reported accuracy is mean accuracy across syndromes, so when
    bootstrap is requested, syndromes are sampled with replacement.
    """

    test_set_representations = [
        np.array([
            test_set_representations[j][i]
            for j in range(len(test_set_representations))
        ])
        for i in range(len(test_set_representations[0]))
    ]

    gallery_set_representations = [
        np.array([
            gallery_set_representations[j][i]
            for j in range(len(gallery_set_representations))
        ])
        for i in range(len(gallery_set_representations[0]))
    ]

    # --------------------------------------------------------
    # Calculate cosine distances.
    #
    # Shape:
    #   [test image, model, gallery image]
    # --------------------------------------------------------

    dists = np.stack([
        pairwise_distances(
            test_set_representations[i],
            gallery_set_representations[i],
            metric='cosine'
        )
        for i in range(len(test_set_representations))
    ], axis=1)

    # --------------------------------------------------------
    # Ensemble prediction.
    #
    # Average cosine distance over all models.
    # This is exactly your original ensemble method.
    # --------------------------------------------------------

    ranked_dists = np.argsort(
        np.mean(dists, axis=1),
        axis=1
    )

    # Last column of gallery_df is synd_id.
    ranked_synds = gallery_df.values[
        ranked_dists
    ][:, :, -1]

    # Remove duplicate syndrome IDs while preserving ranking.
    guessed_all = np.array([
        ranked_synds[i][
            np.sort(
                np.unique(
                    ranked_synds[i],
                    return_index=True
                )[1]
            )
        ]
        for i in range(len(ranked_synds))
    ])

    # --------------------------------------------------------
    # Determine whether each test image is correct at each
    # Top-N level.
    # --------------------------------------------------------

    image_correct = {}

    for n in TOP_NS:
        image_correct[n] = np.array([
            test_synd_ids[idx] in guessed_all[idx, :n]
            for idx in range(len(test_synd_ids))
        ])

    # --------------------------------------------------------
    # Calculate mean-per-syndrome accuracy.
    #
    # Every syndrome receives equal weight, regardless of the
    # number of test images belonging to that syndrome.
    # --------------------------------------------------------

    unique_syndromes = np.unique(test_synd_ids)
    syndrome_accuracy = {}

    acc_per = []

    for n in TOP_NS:
        syndrome_accuracy[n] = {}

        for synd_id in unique_syndromes:
            mask = test_synd_ids == synd_id
            syndrome_accuracy[n][synd_id] = np.mean(
                image_correct[n][mask]
            )

        acc_per.append(
            np.mean(
                list(syndrome_accuracy[n].values())
            )
        )

    # --------------------------------------------------------
    # Bootstrap over syndromes.
    #
    # This is used for Frequent -> Frequent, where there is one
    # fixed test/gallery evaluation.
    # --------------------------------------------------------

    bootstrap_results = None

    if calculate_bootstrap:
        rng = np.random.default_rng(seed)
        n_syndromes = len(unique_syndromes)

        bootstrap_distributions = {
            n: np.empty(n_bootstrap)
            for n in TOP_NS
        }

        for b in range(n_bootstrap):
            sampled_indices = rng.integers(
                0,
                n_syndromes,
                size=n_syndromes
            )

            for n in TOP_NS:
                sampled_values = [
                    syndrome_accuracy[n][
                        unique_syndromes[idx]
                    ]
                    for idx in sampled_indices
                ]

                bootstrap_distributions[n][b] = np.mean(
                    sampled_values
                )

        bootstrap_results = {}

        alpha = 1.0 - confidence

        for i, n in enumerate(TOP_NS):
            distribution = bootstrap_distributions[n]

            lower = np.percentile(
                distribution,
                100 * alpha / 2
            )

            upper = np.percentile(
                distribution,
                100 * (1 - alpha / 2)
            )

            bootstrap_results[n] = {
                "accuracy": acc_per[i],
                "mean": np.mean(distribution),
                "se": np.std(distribution, ddof=1),
                "ci_lower": lower,
                "ci_upper": upper,
                "distribution": distribution
            }

    return acc_per, bootstrap_results


# ============================================================
# Statistics for existing split/bootstrap samples
# ============================================================

def summarize_resampling_results(
    acc_per_list,
    confidence=CONFIDENCE
):
    """
    Summarize results when the rows of acc_per_list are already
    bootstrap/resampling samples.

    Example:
        acc_per_list.shape = [num_splits, 4]

    Columns correspond to:
        Top-1, Top-5, Top-10, Top-30

    No additional bootstrap is performed.

    Returns: Mean, SE and percentile CI for each Top-N.
    """

    acc_per_list = np.asarray(acc_per_list)

    if acc_per_list.ndim != 2 or acc_per_list.shape[1] != 4:
        raise ValueError(
            "acc_per_list must have shape [num_samples, 4]."
        )

    alpha = 1.0 - confidence
    results = {}

    for i, n in enumerate(TOP_NS):
        distribution = acc_per_list[:, i]

        results[n] = {
            "mean": np.mean(distribution),
            "se": np.std(distribution, ddof=1),
            "ci_lower": np.percentile(
                distribution,
                100 * alpha / 2
            ),
            "ci_upper": np.percentile(
                distribution,
                100 * (1 - alpha / 2)
            ),
            "n_samples": len(distribution),
            "distribution": distribution
        }

    return results


# ============================================================
# Print bootstrap results
# ============================================================

def print_bootstrap_results(
    results,
    title="Bootstrap results"
):
    print()
    print(title)
    print("-" * len(title))

    for n in TOP_NS:
        result = results[n]

        print(
            f"Top-{n:2d}: "
            f"Mean = {result['mean'] * 100:.2f}% | "
            f"SE = {result['se'] * 100:.2f}% | "
            f"95% CI = "
            f"[{result['ci_lower'] * 100:.2f}%, "
            f"{result['ci_upper'] * 100:.2f}%]"
        )


# ============================================================
# Load syndrome lookup table
# ============================================================

with open(
    '../lookup_table_gmdb_v1.1.0.txt',
    'r'
) as f:
    line = f.readlines()[1]
    synd_lookup_table = np.array(json.loads(line))


# ============================================================
# Metadata
# ============================================================

data_path = os.path.join('../'
    'data',
    'GestaltMatcherDB',
    'v1.1.0',
    'gmdb_metadata'
)


# ============================================================
# Load ensemble representations
# ============================================================

representation_df = pd.read_csv(
    "all_encodings_non-stratified_GMDB_ethnicity_split_2026.csv",
    delimiter=";"
)


def prep_csv(df, is_pickle=False):
    df = (
        df.groupby('img_name')
        .agg(lambda x: list(x))
        .reset_index()
    )

    if not is_pickle:
        df.representations = df.representations.apply(
            lambda x: np.array([
                json.loads(i)
                for i in x
            ])
        )

    df.img_name = df.img_name.apply(
        lambda x: x.split('_')[0]
    )

    return df


representation_df = prep_csv(
    representation_df,
    is_pickle=False
)


# ============================================================
# TEST 1
# Frequent test / Frequent gallery
#
# Here there is one fixed test/gallery evaluation.
# Therefore we perform a syndrome-level bootstrap.
# ============================================================

gallery_df = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_frequent_gallery_images_v1.1.0.csv'
    )
)

gallery_df['synd_id'] = np.array([
    np.where(
        synd_lookup_table == sid
    )[0][0]
    for sid in gallery_df.label
])

test_df = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_frequent_test_images_v1.1.0.csv'
    )
)

test_synd_ids = np.array([
    np.where(
        synd_lookup_table == sid
    )[0][0]
    for sid in test_df.label
])

gallery_df['image_id'] = gallery_df['image_id'].astype(str)
test_df['image_id'] = test_df['image_id'].astype(str)

gallery_set_representations = (
    representation_df.representations.values[
        np.nonzero(
            gallery_df.image_id.values[:, None]
            ==
            representation_df.img_name.values
        )[1]
    ]
)

test_set_representations = (
    representation_df.representations.values[
        np.nonzero(
            test_df.image_id.values[:, None]
            ==
            representation_df.img_name.values
        )[1]
    ]
)

acc_per, bootstrap_results = eval(
    gallery_df,
    gallery_set_representations,
    test_set_representations,
    test_synd_ids,
    calculate_bootstrap=True,
    n_bootstrap=N_BOOTSTRAP,
    confidence=CONFIDENCE,
    seed=RANDOM_SEED
)

print('===========================================================')
print('---------   test: Frequent, gallery: Frequent    ----------')
print('|Test set     |Gallery |Test  |Top-1 |Top-5 |Top-10|Top-30|')
print('|{}|{}    |{}   |{:.2f} |{:.2f} |{:.2f} |{:.2f} |'.format(
    "GMDB-frequent",
    len(gallery_set_representations),
    len(test_set_representations),
    acc_per[0] * 100,
    acc_per[1] * 100,
    acc_per[2] * 100,
    acc_per[3] * 100
))

print_bootstrap_results(
    bootstrap_results,
    title="Frequent → Frequent: bootstrap results"
)

# Save the actual bootstrap distributions.
pd.DataFrame({
    "top1": bootstrap_results[1]["distribution"],
    "top5": bootstrap_results[5]["distribution"],
    "top10": bootstrap_results[10]["distribution"],
    "top30": bootstrap_results[30]["distribution"]
}).to_csv(
    "bootstrap_frequent_frequent.csv",
    index=False
)


# ============================================================
# TEST 2
# Rare test / Rare gallery
#
# IMPORTANT:
# num_splits are already in resampling/bootstrap samples.
# ============================================================

gallery_df = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_rare_gallery_images_v1.1.0.csv'
    )
)

gallery_df['synd_id'] = np.array([
    sid for sid in gallery_df.label
])

test_df = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_rare_test_images_v1.1.0.csv'
    )
)

gallery_df['image_id'] = gallery_df['image_id'].astype(str)
test_df['image_id'] = test_df['image_id'].astype(str)

acc_per_list = []

num_splits = max(gallery_df.split) + 1

for test_split in range(num_splits):
    test_synd_ids = np.array(
        test_df[
            test_df.split == test_split
        ].label
    )

    gallery_df_split = gallery_df[
        gallery_df.split == test_split
    ]

    gallery_set_representations = (
        representation_df.representations.values[
            np.nonzero(
                gallery_df_split.image_id.values[:, None]
                ==
                representation_df.img_name.values
            )[1]
        ]
    )

    test_set_representations = (
        representation_df.representations.values[
            np.nonzero(
                test_df[
                    test_df.split == test_split
                ].image_id.values[:, None]
                ==
                representation_df.img_name.values
            )[1]
        ]
    )

    acc_per, _ = eval(
        gallery_df_split,
        gallery_set_representations,
        test_set_representations,
        test_synd_ids,
        calculate_bootstrap=False
    )

    acc_per_list.append(acc_per)

acc_per_list = np.array(acc_per_list)

rare_rare_results = summarize_resampling_results(
    acc_per_list,
    confidence=CONFIDENCE
)

print('---------       test: Rare, gallery: Rare        ----------')
print('|Test set     |Gallery |Test  |Top-1 |Top-5 |Top-10|Top-30|')
print('|{}|{}   |{} |{:.2f} |{:.2f} |{:.2f} |{:.2f} |'.format(
    "GMDB-rare",
    len(gallery_df) / num_splits,
    len(test_df) / num_splits,
    rare_rare_results[1]["mean"] * 100,
    rare_rare_results[5]["mean"] * 100,
    rare_rare_results[10]["mean"] * 100,
    rare_rare_results[30]["mean"] * 100
))

print_bootstrap_results(
    rare_rare_results,
    title="Rare → Rare: existing resampling results"
)


# ============================================================
# TEST 3
# Frequent test / Frequent + Rare gallery
#
# The gallery contains different rare-data splits.
# ============================================================

gallery_df1 = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_frequent_gallery_images_v1.1.0.csv'
    )
)

gallery_df2 = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_rare_gallery_images_v1.1.0.csv'
    )
)

gallery_df = pd.concat([
    gallery_df1,
    gallery_df2
])

gallery_df['synd_id'] = np.array([
    sid for sid in gallery_df.label
])

test_df = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_frequent_test_images_v1.1.0.csv'
    )
)

gallery_df['image_id'] = gallery_df['image_id'].astype(str)
test_df['image_id'] = test_df['image_id'].astype(str)

acc_per_list = []

num_splits = max(
    gallery_df.dropna().split
) + 1

for test_split in range(int(num_splits)):
    test_synd_ids = np.array(
        test_df.label
    )

    # Put the frequent gallery into every split.
    # The rare gallery is selected according to the split.
    gallery_df_split = gallery_df.fillna(test_split)

    gallery_df_split = gallery_df_split[
        gallery_df_split.split == test_split
    ].reset_index(drop=True)

    gallery_set_representations = (
        representation_df.representations.values[
            np.nonzero(
                gallery_df_split.image_id.values[:, None]
                ==
                representation_df.img_name.values
            )[1]
        ]
    )

    test_set_representations = (
        representation_df.representations.values[
            np.nonzero(
                test_df.image_id.values[:, None]
                ==
                representation_df.img_name.values
            )[1]
        ]
    )

    acc_per, _ = eval(
        gallery_df_split,
        gallery_set_representations,
        test_set_representations,
        test_synd_ids,
        calculate_bootstrap=False
    )

    acc_per_list.append(acc_per)

acc_per_list = np.array(acc_per_list)

freq_frequence_rare_results = summarize_resampling_results(
    acc_per_list,
    confidence=CONFIDENCE
)

print('--------- test: Frequent, gallery: Frequent+Rare ----------')
print('|Test set     |Gallery |Test  |Top-1 |Top-5 |Top-10|Top-30|')
print('|{}|{}  |{}   |{:.2f} |{:.2f} |{:.2f} |{:.2f} |'.format(
    "GMDB-frequent",
    len(gallery_df1) + len(gallery_df2) / num_splits,
    len(test_df),
    freq_frequence_rare_results[1]["mean"] * 100,
    freq_frequence_rare_results[5]["mean"] * 100,
    freq_frequence_rare_results[10]["mean"] * 100,
    freq_frequence_rare_results[30]["mean"] * 100
))

print_bootstrap_results(
    freq_frequence_rare_results,
    title="Frequent → Frequent+Rare: resampling results"
)

# Save split-level distributions.
pd.DataFrame({
    "top1": freq_frequence_rare_results[1]["distribution"],
    "top5": freq_frequence_rare_results[5]["distribution"],
    "top10": freq_frequence_rare_results[10]["distribution"],
    "top30": freq_frequence_rare_results[30]["distribution"]
}).to_csv(
    "bootstrap_frequent_frequent_rare.csv",
    index=False
)


# ============================================================
# TEST 4
# Rare test / Frequent + Rare gallery
#
# Again, the rare splits are already the resampling samples.
# We therefore report mean, SE and 95% CI over the splits.
# ============================================================

gallery_df1 = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_frequent_gallery_images_v1.1.0.csv'
    )
)

gallery_df2 = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_rare_gallery_images_v1.1.0.csv'
    )
)

gallery_df = pd.concat([
    gallery_df1,
    gallery_df2
])

gallery_df['synd_id'] = np.array([
    sid for sid in gallery_df.label
])

test_df = pd.read_csv(
    os.path.join(
        data_path,
        'gmdb_rare_test_images_v1.1.0.csv'
    )
)

gallery_df['image_id'] = gallery_df['image_id'].astype(str)
test_df['image_id'] = test_df['image_id'].astype(str)

acc_per_list = []

num_splits = max(
    gallery_df.dropna().split
) + 1

for test_split in range(int(num_splits)):
    test_synd_ids = np.array(
        test_df[
            test_df.split == test_split
        ].label
    )

    gallery_df_split = gallery_df.fillna(test_split)

    gallery_df_split = gallery_df_split[
        gallery_df_split.split == test_split
    ].reset_index(drop=True)

    gallery_set_representations = (
        representation_df.representations.values[
            np.nonzero(
                gallery_df_split.image_id.values[:, None]
                ==
                representation_df.img_name.values
            )[1]
        ]
    )

    test_set_representations = (
        representation_df.representations.values[
            np.nonzero(
                test_df[
                    test_df.split == test_split
                ].image_id.values[:, None]
                ==
                representation_df.img_name.values
            )[1]
        ]
    )

    acc_per, _ = eval(
        gallery_df_split,
        gallery_set_representations,
        test_set_representations,
        test_synd_ids,
        calculate_bootstrap=False
    )

    acc_per_list.append(acc_per)

acc_per_list = np.array(acc_per_list)

rare_frequent_rare_results = summarize_resampling_results(
    acc_per_list,
    confidence=CONFIDENCE
)

print('---------   test: Rare, gallery: Frequent+Rare   ----------')
print('|Test set     |Gallery |Test  |Top-1 |Top-5 |Top-10|Top-30|')
print('|{}|{}  |{} |{:.2f} |{:.2f} |{:.2f} |{:.2f} |'.format(
    "GMDB-rare",
    len(gallery_df1) + len(gallery_df2) / num_splits,
    len(test_df) / num_splits,
    rare_frequent_rare_results[1]["mean"] * 100,
    rare_frequent_rare_results[5]["mean"] * 100,
    rare_frequent_rare_results[10]["mean"] * 100,
    rare_frequent_rare_results[30]["mean"] * 100
))

print_bootstrap_results(
    rare_frequent_rare_results,
    title="Rare → Frequent+Rare: existing resampling results"
)


# ============================================================
# Final summary
# ============================================================

print()
print('===========================================================')
print('FINAL STATISTICAL SUMMARY')
print('===========================================================')

summary_rows = []

experiments = [
    (
        "Frequent → Frequent",
        bootstrap_results
    ),
    (
        "Rare → Rare",
        rare_rare_results
    ),
    (
        "Frequent → Frequent+Rare",
        freq_frequence_rare_results
    ),
    (
        "Rare → Frequent+Rare",
        rare_frequent_rare_results
    )
]

for experiment_name, results in experiments:
    for n in TOP_NS:
        result = results[n]

        summary_rows.append({
            "experiment": experiment_name,
            "top_n": n,
            "mean_accuracy": result["mean"],
            "SE": result["se"],
            "CI_lower": result["ci_lower"],
            "CI_upper": result["ci_upper"],
            "mean_accuracy_percent": result["mean"] * 100,
            "SE_percent": result["se"] * 100,
            "CI_lower_percent": result["ci_lower"] * 100,
            "CI_upper_percent": result["ci_upper"] * 100
        })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    "ensemble_bootstrap_summary_non-stratified_ethnicity_Swarm.csv",
    index=False
)

print()
print(summary_df[
    [
        "experiment",
        "top_n",
        "mean_accuracy_percent",
        "SE_percent",
        "CI_lower_percent",
        "CI_upper_percent"
    ]
].to_string(index=False))

print()
print(
    "Summary saved to: ensemble_bootstrap_summary.csv"
)
print(
    "==========================================================="
)


---------   test: Frequent, gallery: Frequent    ----------
|Test set     |Gallery |Test  |Top-1 |Top-5 |Top-10|Top-30|
|GMDB-frequent|8794    |882   |36.91 |57.26 |67.11 |77.91 |

Frequent → Frequent: bootstrap results
--------------------------------------
Top- 1: Mean = 37.01% | SE = 3.16% | 95% CI = [32.18%, 42.46%]
Top- 5: Mean = 57.04% | SE = 3.00% | 95% CI = [52.59%, 60.73%]
Top-10: Mean = 66.95% | SE = 2.71% | 95% CI = [63.15%, 71.26%]
Top-30: Mean = 78.15% | SE = 0.96% | 95% CI = [76.52%, 79.04%]
---------       test: Rare, gallery: Rare        ----------
|Test set     |Gallery |Test  |Top-1 |Top-5 |Top-10|Top-30|
|GMDB-rare|922.6   |386.4 |31.10 |45.08 |51.76 |66.95 |

Rare → Rare: existing resampling results
----------------------------------------
Top- 1: Mean = 31.10% | SE = 1.53% | 95% CI = [28.02%, 32.74%]
Top- 5: Mean = 45.08% | SE = 1.93% | 95% CI = [41.06%, 46.91%]
Top-10: Mean = 51.76% | SE = 1.95% | 95% CI = [47.64%, 53.39%]
Top-30: Mean = 66.95% | SE = 1.18% | 95% 